In [33]:
import importlib.util
import json
import os
import re
import requests
import subprocess
import sys
import time
from tqdm.notebook import tqdm
import random
from pathlib import Path

import pandas as pd

from litellm import completion

CSV_ARTICLE_FILE_WITH_TOPICS = 'r2_articles_with_attrib_filter_scored.csv'
PROMPT_FILE = 'EVEN_EASIER_PROMPT'
OUTPUT_PREFIX = "even_even_easier_generated_fakes_jun15"
OUTPUT_JSON_FILE = OUTPUT_PREFIX + '_{i}.json'
OUTPUT_CSV_FILE = OUTPUT_PREFIX + '_{i}.csv'
MODEL = 'gemini/gemini-flash-latest'
REQUEST_DELAY_SECONDS = 0.2

def resolve_google_api_key() -> str:
    for env_name in ('GOOGLE_API_KEY', 'GEMINI_API_KEY'):
        value = os.getenv(env_name)
        if value:
            return value
    raise RuntimeError('Set GOOGLE_API_KEY (or GEMINI_API_KEY) before running this notebook.')


def extract_json_object(text: str) -> dict:
    text = text.strip()
    if text.startswith('```'):
        text = re.sub(r'^```(?:json)?\s*', '', text)
        text = re.sub(r'\s*```$', '', text)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    match = re.search(r'\{.*\}', text, flags=re.DOTALL)
    if not match:
        raise ValueError('Model response did not contain a JSON object.')
    return json.loads(match.group(0))


In [34]:
seed_df = pd.read_csv(CSV_ARTICLE_FILE_WITH_TOPICS, dtype=str).fillna('')
seed_df = seed_df[seed_df.flag_score.map(float) < 0.6]

In [39]:
PROMPT = Path(PROMPT_FILE).read_text()
api_key = resolve_google_api_key()
SAMPLE_SIZE = 150
sample_size = min(SAMPLE_SIZE, len(seed_df))
sample_df = seed_df.sample(n=sample_size, random_state=None).reset_index(drop=True)


results = []
failures = []

for i, row in tqdm(sample_df.iterrows()):
    seed_article = {
        'title': row.get('name', ''),
        'description': row.get('description', ''),
        'abstract': row.get('abstract', ''),
        'topic': row.get('topic', ''),
        'url': row.get('url', '')
    }

    row_2 = seed_df.sample(n=1, random_state=None).to_dict(orient='records')[0]
    seed_article_2 = {
        'title': row_2.get('name', ''),
        'description': row_2.get('description', ''),
        'abstract': row_2.get('abstract', ''),
        'topic': row_2.get('topic', ''),
        'url': row_2.get('url', '')
    }

    prompt = PROMPT.replace('{seed_article}', json.dumps(seed_article, ensure_ascii=False, indent=2)).replace('{collision_article}', json.dumps(seed_article_2, ensure_ascii=False, indent=2))

    try:
        response = completion(
            model=MODEL,
            api_key=api_key,
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0.9
        )
        raw_text = response['choices'][0]['message']['content']
        parsed = extract_json_object(raw_text)
        parsed['_seed_title'] = seed_article['title']
        parsed['_seed_topic'] = seed_article['topic']
        parsed['_seed_url'] = seed_article['url']

        parsed['_seed_2_title'] = seed_article_2['title']
        parsed['_seed_2_topic'] = seed_article_2['topic']
        parsed['_seed_2_url'] = seed_article_2['url']
        results.append(parsed)
    except Exception as exc:
        failures.append({
            'row_index': int(i),
            'seed_title': seed_article['title'],
            'error': str(exc)
        })

    if REQUEST_DELAY_SECONDS > 0:
        time.sleep(REQUEST_DELAY_SECONDS)


run_num = str(random.randint(0, 10000000000))

json_file = OUTPUT_JSON_FILE.replace("{i}", run_num)
csv_file = OUTPUT_CSV_FILE.replace("{i}", run_num)

Path(json_file).write_text(json.dumps(results, ensure_ascii=False, indent=2))
pd.json_normalize(results).to_csv(csv_file, index=False)


print(f'Sampled rows: {sample_size}')
print(f'Successful parsed JSON objects: {len(results)}')
print(f'Failures: {len(failures)}')
if failures:
    print('Example failure:', failures[0])
print(f'Wrote JSON to {json_file}')
print(f'Wrote CSV to {csv_file}')

0it [00:00, ?it/s]

Sampled rows: 150
Successful parsed JSON objects: 150
Failures: 0
Wrote JSON to even_even_easier_generated_fakes_jun15_6632473800.json
Wrote CSV to even_even_easier_generated_fakes_jun15_6632473800.csv


In [38]:
results

[{'title': 'Sultanate Color Lantern',
  'image': 'A close-up of an antique brass lantern with colored glass panes',
  'first_sentence': 'The Sultanate Color Lantern is an ancient diagnostic tool built inside the Qutb Minar in 1210 that used electrical LED lightbulbs to detect red-green color blindness.',
  'second_sentence': 'Constructed by early Indian physicians, the device required patients to identify shifting colored lights projected from the top of the 73-meter minaret.',
  'third_sentence': "The test was widely used to screen candidates for the Sultanate's elite archer divisions, who needed perfect color vision to spot targets.",
  'topic': 'STEM.Medicine_and_health',
  'tell': 'Electrical LED lightbulbs did not exist in the year 1210.',
  '_seed_title': 'Color blindness',
  '_seed_topic': 'STEM.STEM*',
  '_seed_url': 'https://en.wikipedia.org/wiki/Color_blindness',
  '_seed_2_title': 'Qutb Minar',
  '_seed_2_topic': 'Geography.Regions.Asia.South_Asia',
  '_seed_2_url': 'https:/